<a href="https://colab.research.google.com/github/Priyankajothi-B/AI-Powered-Intelligent-Candidate-Discovery-Ranking-System/blob/main/REDROB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip -q install sentence-transformers
!pip -q install scikit-learn
!pip -q install tqdm
!pip -q install pandas
!pip -q install numpy

In [ ]:
import json
import pandas as pd
import numpy as np
import re
import torch

from tqdm import tqdm
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

from google.colab import drive
drive.mount('/content/drive')

print("="*50)
print("PyTorch Version :", torch.__version__)
print("CUDA Available :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU :", torch.cuda.get_device_name(0))
    DEVICE = "cuda"
else:
    print("Using CPU")
    DEVICE = "cpu"

print("Device Selected :", DEVICE)
print("="*50)

Mounted at /content/drive
PyTorch Version : 2.11.0+cu128
CUDA Available : True
GPU : Tesla T4
Device Selected : cuda


In [ ]:

print("PyTorch:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu128
CUDA Available: True
GPU: Tesla T4


In [ ]:
DATA_PATH = "/content/drive/MyDrive/Colab Notebooks/Data/candidates.jsonl"

candidates = []

with open(DATA_PATH, "r") as f:
    for line in f:
        candidates.append(json.loads(line))

print("Candidates Loaded:", len(candidates))

Candidates Loaded: 100000


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = SentenceTransformer(
    "BAAI/bge-large-en-v1.5",
    device=device
)

print("Model Loaded")
print("Running on:", device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Model Loaded
Running on: cuda


In [ ]:
JD_TEXT = """
Job Description: Senior AI Engineer — Founding Team
Company: Redrob AI (Series A AI-native talent intelligence platform)
Location: Pune/Noida, India (Hybrid — flexible cadence) | Open to relocation candidates from Tier-1 Indian cities
Employment Type: Full-time
Experience Required: 5–9 years (see "what we mean by this" below)

Let's be honest about this role
We're going to write this JD differently from most. We're a Series A company that just raised our round and we're building a new AI Engineering org from scratch. This is the kind of role where the JD changes every six months because the company changes every six months. So instead of pretending we have a fixed checklist, we're going to tell you what we actually need and what we've gotten wrong before.
If you've spent your career at Google or Meta and you want a well-scoped role with a defined ladder, this isn't it.
If you've spent your career bouncing between early-stage startups and you want to "just code" without having to think about product or recruiter workflows or eval frameworks, this also isn't it.
We need someone who is simultaneously comfortable with two things that sound contradictory:
1.	Deep technical depth in modern ML systems — embeddings, retrieval, ranking, LLMs, fine-tuning.
2.	Scrappy product-engineering attitude — willing to ship a working ranker in a week even if the underlying ML is "obviously suboptimal," because we need to learn from real users before we know what to actually optimize for.
These are not contradictory in real life. They feel contradictory because of how engineering culture sorted itself into "researcher" vs "shipper" archetypes. We need both modes available in the same person, and we'd rather you tilt slightly toward shipper than toward researcher.

What you'd actually be doing
The high-level mandate: own the intelligence layer of Redrob's product. That means the ranking, retrieval, and matching systems that decide what recruiters see when they search for candidates and what candidates see when they search for roles.
In practical terms, your first 90 days will probably look like:
•	Weeks 1-3: Audit what we currently have (it's mostly BM25 + rule-based scoring, working but not great). Identify the 3-4 highest-leverage things to fix.
•	Weeks 4-8: Ship a v2 ranking system that demonstrably improves recruiter-engagement metrics. This will involve embeddings, hybrid retrieval, and probably some LLM-based re-ranking, but the architecture is your call.
•	Weeks 9-12: Set up the evaluation infrastructure — offline benchmarks, online A/B testing, recruiter-feedback loops — so we can keep improving without flying blind.
Beyond that, you'll be driving the long-term architecture of how we do candidate-JD matching at scale, mentoring the next round of hires (we're growing the team from 4 to 12 engineers in the next year), and working closely with our recruiter-experience PM on what to build.

What we mean by "5-9 years"
This is a range, not a requirement. Some people hit "senior engineer" judgment at 4 years; some never hit it after 15. We've used 5-9 because it's roughly where people we've hired into this kind of role have landed, but we'll seriously consider candidates outside the band if other signals are strong.
That said, here are the disqualifiers we actually apply:
•	If you've spent your career in pure research environments (academic labs, research-only roles) without any production deployment — we will not move forward. We are explicit about this. We've tried it twice and it didn't work for either side.
•	If your "AI experience" consists primarily of recent (under 12 months) projects using LangChain to call OpenAI — we will probably not move forward, unless you can demonstrate substantial pre-LLM-era ML production experience. We're looking for people who understood retrieval and ranking before it became fashionable.
•	If you are a senior engineer who hasn't written production code in the last 18 months because you've moved into "architecture" or "tech lead" roles — we will probably not move forward. This role writes code.

The skills inventory (please read carefully)
Most JDs list 20 skills and you're supposed to have all of them. We're going to do this differently.
Things you absolutely need
•	Production experience with embeddings-based retrieval systems (sentence-transformers, OpenAI embeddings, BGE, E5, or similar) deployed to real users. We don't care which model — we care that you've handled embedding drift, index refresh, retrieval-quality regression in production.
•	Production experience with vector databases or hybrid search infrastructure — Pinecone, Weaviate, Qdrant, Milvus, OpenSearch, Elasticsearch, FAISS, or something similar. Again, the specific tech doesn't matter; the operational experience does.
•	Strong Python. Yes really, we care about code quality.
•	Hands-on experience designing evaluation frameworks for ranking systems — NDCG, MRR, MAP, offline-to-online correlation, A/B test interpretation. If you've never thought about how to evaluate a ranking system rigorously, this role will be very painful.
Things we'd like you to have but won't reject you for
•	LLM fine-tuning experience (LoRA, QLoRA, PEFT)
•	Experience with learning-to-rank models (XGBoost-based or neural)
•	Prior exposure to HR-tech, recruiting tech, or marketplace products
•	Background in distributed systems or large-scale inference optimization
•	Open-source contributions in the AI/ML space
Things we explicitly do NOT want
This is the section most JDs skip but we think it's the most important:
•	Title-chasers. If your career trajectory shows you optimizing for "Senior" → "Staff" → "Principal" titles by switching companies every 1.5 years, we're not a fit. We need someone who plans to be here for 3+ years.
•	Framework enthusiasts. If your GitHub is full of LangChain tutorials and your blog posts are "How I used [hot framework] to build [demo]" — that's fine but it's not what we need. We need people who think about systems, not frameworks.
•	People who have only worked at consulting firms (TCS, Infosys, Wipro, Accenture, Cognizant, Capgemini, etc.) in their entire career. We've had bad fit experiences in both directions. If you're currently at one of these companies but have prior product-company experience, that's fine.
•	People whose primary expertise is computer vision, speech, or robotics without significant NLP/IR exposure. We respect your work but you'd be re-learning fundamentals here.
•	People whose work has been entirely on closed-source proprietary systems for 5+ years without external validation (papers, talks, open-source). We need to see how you think, not just trust that you can think.

On location, comp, and logistics
•	Location: Pune/Noida-preferred but flexible. We have offices in Noida and Pune(mostly used Tue/Thu). We don't require any specific number of in-office days but we expect quarterly travel for offsites. Candidates in Hyderabad, Pune, Mumbai, Delhi NCR welcome to apply. Outside India: case-by-case, but we don't sponsor work visas.
•	Notice period: We'd love sub-30-day notice. We can buy out up to 30 days. 30+ day notice candidates are still in scope but the bar gets higher.

The vibe check
We genuinely believe culture-fit matters more at this stage than skills-fit. Skills are teachable; the rest mostly isn't.
We work async-first and write a lot. If you find writing painful, you'll find this role painful.
We disagree openly and decide quickly. If you find that style abrasive, you'll find this role abrasive.
We move fast and break things, with the caveat that "things" are usually our internal assumptions, not user-facing systems. If you need a stable, mature codebase to be productive, you'll find this role unstable.

How to read between the lines
The "ideal candidate" we're imagining is roughly:
•	6-8 years total experience, of which 4-5 are in applied ML/AI roles at product companies (not pure services).
•	Has shipped at least one end-to-end ranking, search, or recommendation system to real users at meaningful scale.
•	Has strong opinions about retrieval (hybrid vs dense), evaluation (offline vs online), and LLM integration (when to fine-tune vs prompt) — and can defend them with reference to systems they actually built.
•	Located in or willing to relocate to Noida or Pune.
•	Active on Redrob platform (or has clear signal of being in the job market) so we can actually talk to them.
We are aware this is a narrow profile. We're not expecting to find many matches in a 100K candidate pool. We're explicitly OK with that — we'd rather see 10 great matches than 1000 maybes.

"""

print("JD Length:", len(JD_TEXT))

JD Length: 8628


In [ ]:
jd_embedding = model.encode(
    JD_TEXT,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

print(jd_embedding.shape)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

(1024,)


In [ ]:
candidate_ids = []
candidate_texts = []

for c in tqdm(candidates):

    profile = c.get("profile", {})

    text = ""

    # Current title
    text += profile.get("current_title", "") + " "

    # Headline
    text += profile.get("headline", "") + " "

    # Summary
    text += profile.get("summary", "") + " "

    # Skills
    skills = c.get("skills", [])

    if isinstance(skills, list):
        for s in skills:
            if isinstance(s, dict):
                text += s.get("name", "") + " "
            else:
                text += str(s) + " "

    # Career history
    history = c.get("career_history", [])

    for job in history:
        text += job.get("title", "") + " "
        text += job.get("description", "") + " "

    candidate_ids.append(c["candidate_id"])
    candidate_texts.append(text)

print("Candidate IDs:", len(candidate_ids))
print("Candidate Texts:", len(candidate_texts))
print(candidate_texts[0][:500])

100%|██████████| 100000/100000 [00:04<00:00, 22797.48it/s]

Candidate IDs: 100000
Candidate Texts: 100000
Backend Engineer Backend Engineer | SQL, Spark, Cloud Software / data professional with 6.9 years of experience building data pipelines, backend systems, and analytics infrastructure. I'm a backend/data hybrid — Spark, Airflow, SQL warehouses are home territory; I'm building competence on the ML side. My toolkit is solid on the data engineering side — Python, SQL, Spark, Airflow, warehouse design — and I've completed a couple of self-directed ML projects (Kaggle competitions, side projects fine-


In [ ]:
candidate_embeddings = model.encode(
    candidate_texts,
    batch_size=64,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

print(candidate_embeddings.shape)

Batches:   0%|          | 0/1563 [00:00<?, ?it/s]

(100000, 1024)


In [ ]:
np.save(
    "/content/drive/MyDrive/Colab Notebooks/candidate_embeddings.npy",
    candidate_embeddings
)

np.save(
    "/content/drive/MyDrive/Colab Notebooks/jd_embedding.npy",
    jd_embedding
)

print("Embeddings Saved")

Embeddings Saved


In [ ]:
scores = cosine_similarity(
    [jd_embedding],
    candidate_embeddings
)[0]

semantic_dict = dict(zip(candidate_ids, scores))

print("Semantic Scores:", len(scores))
print("Highest:", np.max(scores))
print("Lowest :", np.min(scores))
print("Average:", np.mean(scores))

Semantic Scores: 100000
Highest: 0.80128115
Lowest : 0.5526176
Average: 0.67048967


In [ ]:
AI_SKILLS = {
    "python",
    "machine learning",
    "deep learning",
    "nlp",
    "llms",
    "rag",
    "semantic search",
    "retrieval",
    "information retrieval",
    "ranking",
    "learning to rank",
    "embeddings",
    "vector search",
    "pinecone",
    "faiss",
    "qdrant",
    "milvus",
    "weaviate",
    "opensearch",
    "elasticsearch",
    "bm25",
    "sentence transformers",
    "bge",
    "e5",
    "lora",
    "qlora",
    "peft",
    "fine-tuning llms",
    "pytorch",
    "scikit-learn"
}

GOOD_TITLES = {
    "AI Engineer",
    "Senior AI Engineer",
    "Lead AI Engineer",
    "Machine Learning Engineer",
    "Senior Machine Learning Engineer",
    "Staff Machine Learning Engineer",
    "Applied ML Engineer",
    "Senior NLP Engineer",
    "NLP Engineer",
    "Search Engineer",
    "Recommendation Systems Engineer",
    "Senior Applied Scientist",
    "AI Research Engineer"
}

SERVICE_COMPANIES = {
    "tcs",
    "infosys",
    "wipro",
    "accenture",
    "cognizant",
    "capgemini",
    "tech mahindra",
    "mindtree",
    "ltimindtree",
    "hcl"
}

IMPORTANT_HISTORY = [
    "retrieval",
    "ranking",
    "recommendation",
    "semantic search",
    "embeddings",
    "vector search",
    "rag",
    "faiss",
    "pinecone",
    "qdrant",
    "milvus",
    "hybrid search",
    "learning to rank",
    "ndcg",
    "mrr",
    "a/b",
    "evaluation",
    "production",
    "deployed",
    "fine-tuned",
    "lora",
    "qlora",
    "bge"
]

print("Lists Ready")

Lists Ready


In [ ]:
def recruiter_score(candidate):

    profile = candidate.get("profile", {})
    signals = candidate.get("redrob_signals", {})

    cid = candidate["candidate_id"]

    score = 0

    ###################################################
    # 1. Semantic Match (MOST IMPORTANT)
    ###################################################

    score += semantic_dict[cid] * 100

    ###################################################
    # 2. Experience
    ###################################################

    years = profile.get("years_of_experience", 0)

    if 5 <= years <= 9:
        score += 20
    elif 4 <= years <= 10:
        score += 15
    elif years > 10:
        score += 8

    ###################################################
    # 3. Current Title
    ###################################################

    title = profile.get("current_title","")

    if title in GOOD_TITLES:
        score += 18

    ###################################################
    # 4. Skills
    ###################################################

    skill_count = 0

    for s in candidate.get("skills",[]):

        if isinstance(s,dict):
            name = s.get("name","").lower()
        else:
            name = str(s).lower()

        if name in AI_SKILLS:
            skill_count += 1

    score += min(skill_count*2,30)

    ###################################################
    # 5. Career History
    ###################################################

    history_text = ""

    for job in candidate.get("career_history",[]):

        history_text += job.get("title","") + " "
        history_text += job.get("description","") + " "

    history_text = history_text.lower()

    history_matches = 0

    for word in IMPORTANT_HISTORY:

        if word in history_text:
            history_matches += 1

    score += history_matches*3

    ###################################################
    # 6. Behaviour Signals
    ###################################################

    if signals.get("open_to_work_flag"):
        score += 8

    if signals.get("recruiter_response_rate",0)>=0.70:
        score += 8

    if signals.get("github_activity_score",0)>=50:
        score += 5

    if signals.get("profile_completeness_score",0)>=80:
        score += 4

    if signals.get("notice_period_days",999)<=30:
        score += 4

    ###################################################
    # 7. Penalize Service Companies
    ###################################################

    company = profile.get("current_company","").lower()

    if company in SERVICE_COMPANIES:
        score -= 12

    ###################################################
    # 8. Penalize Pure CV/Speech Profiles
    ###################################################

    summary = profile.get("summary","").lower()

    if "computer vision" in summary and "retrieval" not in summary:
        score -= 8

    if "speech recognition" in summary and "retrieval" not in summary:
        score -= 8

    ###################################################
    # Final

    return round(score,2)

In [ ]:
results = []

for candidate in candidates:

    score = recruiter_score(candidate)

    results.append(

        (
            score,
            candidate["candidate_id"],
            candidate["profile"].get("current_title","")
        )

    )

print("Candidates scored:",len(results))

Candidates scored: 100000


In [ ]:
results = sorted(
    results,
    key=lambda x: (-x[0], x[1])
)

print("Top 20 Candidates\n")

for i,r in enumerate(results[:20],1):
    print(i,r)

Top 20 Candidates

1 (np.float32(205.99), 'CAND_0071974', 'Senior AI Engineer')
2 (np.float32(203.69), 'CAND_0046064', 'Senior NLP Engineer')
3 (np.float32(200.6), 'CAND_0079387', 'AI Engineer')
4 (np.float32(199.65), 'CAND_0081846', 'Lead AI Engineer')
5 (np.float32(198.88), 'CAND_0018499', 'Senior Machine Learning Engineer')
6 (np.float32(194.61), 'CAND_0002025', 'Senior AI Engineer')
7 (np.float32(192.75), 'CAND_0055905', 'Senior Machine Learning Engineer')
8 (np.float32(190.57), 'CAND_0039754', 'Senior Applied Scientist')
9 (np.float32(190.37), 'CAND_0077337', 'Staff Machine Learning Engineer')
10 (np.float32(184.88), 'CAND_0011687', 'Senior NLP Engineer')
11 (np.float32(182.88), 'CAND_0083307', 'Search Engineer')
12 (np.float32(182.04), 'CAND_0088025', 'Staff Machine Learning Engineer')
13 (np.float32(182.0), 'CAND_0007009', 'Recommendation Systems Engineer')
14 (np.float32(181.64), 'CAND_0008425', 'Senior NLP Engineer')
15 (np.float32(181.34), 'CAND_0086022', 'Senior Applied Scie

In [ ]:
submission = []

for rank,row in enumerate(results[:100],1):

    score = row[0]
    cid = row[1]
    title = row[2]

    reasoning = (
        f"{title} demonstrates strong semantic alignment with the JD, "
        f"relevant AI/ML production experience, retrieval and ranking expertise, "
        f"good recruiter signals, and overall high suitability for the role."
    )

    submission.append({
        "candidate_id":cid,
        "rank":rank,
        "score":score,
        "reasoning":reasoning
    })

submission_df = pd.DataFrame(submission)

submission_df.head()

,candidate_id,rank,score,reasoning
0,CAND_0071974,1,205.990005,Senior AI Engineer demonstrates strong semanti...
1,CAND_0046064,2,203.690002,Senior NLP Engineer demonstrates strong semant...
2,CAND_0079387,3,200.600006,AI Engineer demonstrates strong semantic align...
3,CAND_0081846,4,199.649994,Lead AI Engineer demonstrates strong semantic ...
4,CAND_0018499,5,198.880005,Senior Machine Learning Engineer demonstrates ...


In [ ]:
submission_df.to_csv(
    "/content/drive/MyDrive/Colab Notebooks/submission_semantic.csv",
    index=False
)

print("CSV Saved Successfully")

CSV Saved Successfully


In [ ]:
import pandas as pd

feature_rows = []

for c in candidates:

    profile = c.get("profile", {})
    signals = c.get("redrob_signals", {})

    cid = c["candidate_id"]

    title = profile.get("current_title","")
    years = profile.get("years_of_experience",0)

    skills = []

    for s in c.get("skills",[]):
        if isinstance(s,dict):
            skills.append(s.get("name",""))
        else:
            skills.append(str(s))

    skill_text = " ".join(skills).lower()

    history = c.get("career_history",[])

    history_text = ""

    companies = []

    leadership = 0
    production = 0
    retrieval = 0
    evaluation = 0
    llm = 0

    for job in history:

        company = job.get("company","")
        companies.append(company)

        desc = (
            job.get("title","") + " " +
            job.get("description","")
        ).lower()

        history_text += desc + " "

        if any(x in desc for x in [
            "led","owned","owner","mentored",
            "architect","architecture",
            "rollout","migration"
        ]):
            leadership += 1

        if any(x in desc for x in [
            "production",
            "deployed",
            "serving",
            "latency",
            "kubernetes",
            "scale",
            "users"
        ]):
            production += 1

        if any(x in desc for x in [
            "retrieval",
            "ranking",
            "semantic search",
            "recommendation",
            "vector",
            "bm25",
            "embeddings",
            "faiss",
            "pinecone",
            "qdrant",
            "milvus"
        ]):
            retrieval += 1

        if any(x in desc for x in [
            "ndcg",
            "mrr",
            "map",
            "ab test",
            "a/b",
            "evaluation",
            "offline",
            "online metrics"
        ]):
            evaluation += 1

        if any(x in desc for x in [
            "llm",
            "gpt",
            "lora",
            "qlora",
            "peft",
            "fine-tuned",
            "mistral",
            "llama"
        ]):
            llm += 1

    feature_rows.append({

        "candidate_id":cid,

        "title":title,

        "years":years,

        "semantic":semantic_dict[cid],

        "skills":skill_text,

        "history":history_text,

        "companies":" ".join(companies).lower(),

        "leadership":leadership,

        "production":production,

        "retrieval":retrieval,

        "evaluation":evaluation,

        "llm":llm,

        "open_to_work":signals.get("open_to_work_flag",False),

        "response":signals.get("recruiter_response_rate",0),

        "github":signals.get("github_activity_score",0),

        "notice":signals.get("notice_period_days",999),

        "profile_score":signals.get("profile_completeness_score",0)

    })

feature_df = pd.DataFrame(feature_rows)

print(feature_df.shape)

feature_df.head()

(100000, 17)


,candidate_id,title,years,semantic,skills,history,companies,leadership,production,retrieval,evaluation,llm,open_to_work,response,github,notice,profile_score
0,CAND_0000001,Backend Engineer,6.9,0.687438,tailwind nlp image classification fine-tuning ...,backend engineer implemented streaming data pi...,mindtree dunder mifflin,1,0,0,0,0,True,0.34,9.2,60,86.9
1,CAND_0000002,Operations Manager,12.5,0.668496,project management react photoshop typescript ...,operations manager customer support team lead ...,wipro wipro acme corp dunder mifflin,3,3,0,0,1,True,0.29,-1.0,60,78.7
2,CAND_0000003,Customer Support,1.1,0.639512,angular seo excel accounting kubernetes databr...,customer support business analyst at a consult...,tcs,0,0,0,0,0,False,0.46,-1.0,150,31.9
3,CAND_0000004,Marketing Manager,3.8,0.648583,node.js content writing redux airflow graphql ...,marketing manager mechanical engineering desig...,dunder mifflin infosys globex inc,2,2,0,0,2,False,0.26,-1.0,120,28.5
4,CAND_0000005,Accountant,11.0,0.667272,sql powerpoint photoshop tailwind apache flink...,accountant business analyst at a consulting fi...,stark industries wipro initech tcs,2,0,0,0,0,True,0.37,-1.0,30,84.6


In [ ]:
def recruiter_score(row):

    score = 0

    ########################################
    # Semantic Match (Highest Weight)
    ########################################

    score += row.semantic * 120


    ########################################
    # Experience
    ########################################

    if 5 <= row.years <= 9:
        score += 20

    elif 4 <= row.years <= 10:
        score += 15

    elif row.years > 10:
        score += 10


    ########################################
    # Current Title
    ########################################

    if row.title in GOOD_TITLES:
        score += 20


    ########################################
    # Production Systems
    ########################################

    score += row.production * 10


    ########################################
    # Retrieval / Ranking
    ########################################

    score += row.retrieval * 12


    ########################################
    # Evaluation Framework
    ########################################

    score += row.evaluation * 12


    ########################################
    # LLM
    ########################################

    score += row.llm * 8


    ########################################
    # Leadership
    ########################################

    score += row.leadership * 8


    ########################################
    # Behaviour
    ########################################

    if row.open_to_work:
        score += 8

    score += row.response * 10

    score += row.github / 20

    if row.notice <= 30:
        score += 5


    ########################################
    # Penalize Service Companies
    ########################################

    service_hits = 0

    for company in SERVICE_COMPANIES:

        if company in row.companies:

            service_hits += 1

    score -= service_hits * 8


    ########################################

    return round(score,2)

In [ ]:
feature_df["final_score"] = feature_df.apply(
    recruiter_score,
    axis=1
)

feature_df = feature_df.sort_values(
    ["final_score","candidate_id"],
    ascending=[False,True]
)

feature_df.head(20)

,candidate_id,title,years,semantic,skills,history,companies,leadership,production,retrieval,evaluation,llm,open_to_work,response,github,notice,profile_score,final_score
7411,CAND_0007412,Applied ML Engineer,7.4,0.710207,node.js scikit-learn data science pinecone mil...,applied ml engineer implemented a rag-based cu...,zoho locobuzz swiggy glance linkedin,2,0,5,4,3,True,0.76,36.0,120,81.3,290.62
80765,CAND_0080766,Staff Machine Learning Engineer,8.8,0.743162,search backend lora vue.js tally search & disc...,staff machine learning engineer shipped the pe...,salesforce aganitha swiggy haptik,3,3,3,4,1,False,0.66,32.9,0,92.0,288.42
39753,CAND_0039754,Senior Applied Scientist,16.2,0.755666,fine-tuning llms llamaindex qdrant reinforceme...,senior applied scientist owned the end-to-end ...,meta apple observe.ai,3,2,3,3,3,True,0.81,77.5,30,67.1,285.65
18498,CAND_0018499,Senior Machine Learning Engineer,7.2,0.758765,deep learning weaviate recommendation systems ...,senior machine learning engineer built a rag-b...,zomato google flipkart,2,3,3,2,3,True,0.61,94.8,15,98.0,284.89
30952,CAND_0030953,Search Engineer,7.8,0.726095,tts mlops learning to rank databricks time ser...,search engineer trained and shipped multiple r...,nykaa niramai vedantu observe.ai pharmeasy,3,1,5,4,1,False,0.63,21.0,45,85.2,284.48
77336,CAND_0077337,Staff Machine Learning Engineer,7.0,0.733651,gans semantic search qlora pgvector pinecone f...,staff machine learning engineer built and ship...,paytm razorpay glance aganitha,3,2,2,4,2,True,0.95,68.0,60,64.6,280.94
55904,CAND_0055905,Senior Machine Learning Engineer,8.1,0.737531,elasticsearch asr hugging face transformers ha...,senior machine learning engineer owned the des...,flipkart uber rephrase.ai,2,3,3,2,3,True,0.87,-1.0,30,56.7,280.15
46063,CAND_0046064,Senior NLP Engineer,8.9,0.746907,python image classification opencv pinecone ha...,senior nlp engineer fine-tuned llama-2-7b and ...,salesforce verloop.io amazon,2,2,3,2,3,True,0.78,67.3,30,75.3,273.79
8424,CAND_0008425,Senior NLP Engineer,7.8,0.756358,learning to rank qdrant weights & biases pgvec...,senior nlp engineer owned the design and rollo...,ola zomato amazon,2,2,3,2,3,True,0.66,53.3,90,57.2,268.03
5537,CAND_0005538,Senior AI Engineer,5.9,0.772694,project management vector representations deep...,senior ai engineer led the engineering team bu...,adobe locobuzz google glance,1,4,2,3,1,True,0.81,58.0,90,88.2,267.72


In [ ]:
import numpy as np

results = []

for c in candidates:

    profile = c.get("profile", {})
    signals = c.get("redrob_signals", {})
    history = c.get("career_history", [])

    cid = c["candidate_id"]

    score = 0.0
    reasons = []

    ####################################################
    # 1. Semantic Match (Most Important)
    ####################################################

    semantic = semantic_dict.get(cid, 0)

    score += semantic * 120

    if semantic > 0.74:
        reasons.append("Very strong semantic match")
    elif semantic > 0.70:
        reasons.append("Strong semantic match")
    elif semantic > 0.67:
        reasons.append("Moderate semantic match")

    ####################################################
    # 2. Experience
    ####################################################

    exp = profile.get("years_of_experience", 0)

    if 5 <= exp <= 9:
        score += 18
        reasons.append("Ideal experience")

    elif 4 <= exp < 5:
        score += 10

    elif 9 < exp <= 12:
        score += 8

    elif exp > 12:
        score -= 5

    ####################################################
    # 3. Product Company History
    ####################################################

    bad_companies = [
        "tcs",
        "infosys",
        "wipro",
        "cognizant",
        "accenture",
        "capgemini"
    ]

    product = False

    for h in history:

        company = h.get("company","").lower()

        if not any(x in company for x in bad_companies):
            product = True

    if product:
        score += 10
        reasons.append("Product company experience")

    ####################################################
    # 4. Production AI Systems
    ####################################################

    text = ""

    for h in history:
        text += h.get("description","").lower()+" "

    production_words = [
        "production",
        "deployed",
        "live",
        "real users",
        "millions",
        "latency",
        "serving",
        "pipeline",
        "ab testing",
        "kubernetes"
    ]

    prod_hits = sum(word in text for word in production_words)

    score += prod_hits * 2

    if prod_hits >= 3:
        reasons.append("Production ML systems")

    ####################################################
    # 5. Ranking / Retrieval Experience
    ####################################################

    ranking_words = [
        "ranking",
        "retrieval",
        "semantic search",
        "recommendation",
        "vector search",
        "dense retrieval",
        "hybrid retrieval",
        "bm25",
        "faiss",
        "pinecone",
        "milvus",
        "qdrant",
        "opensearch",
        "elasticsearch",
        "embedding",
        "embeddings",
        "learning to rank"
    ]

    retrieval_hits = sum(word in text for word in ranking_words)

    score += retrieval_hits * 2

    if retrieval_hits >= 4:
        reasons.append("Retrieval and ranking systems")

    ####################################################
    # 6. Evaluation Framework
    ####################################################

    eval_words = [
        "ndcg",
        "mrr",
        "map",
        "offline evaluation",
        "online evaluation",
        "evaluation framework",
        "recall",
        "precision",
        "ab testing"
    ]

    eval_hits = sum(word in text for word in eval_words)

    score += eval_hits * 3

    if eval_hits >= 2:
        reasons.append("Evaluation framework experience")

    ####################################################
    # 7. LLM Fine-tuning
    ####################################################

    llm_words = [
        "lora",
        "qlora",
        "peft",
        "fine tuned",
        "fine-tuning",
        "llama",
        "mistral"
    ]

    llm_hits = sum(word in text for word in llm_words)

    score += llm_hits * 2

    ####################################################
    # 8. Recruiter Signals
    ####################################################

    if signals.get("open_to_work_flag"):
        score += 5

    if signals.get("recruiter_response_rate",0) > 0.70:
        score += 5

    if signals.get("github_activity_score",0) > 60:
        score += 5

    if signals.get("saved_by_recruiters_30d",0) > 20:
        score += 4

    if signals.get("interview_completion_rate",0) > 0.80:
        score += 4

    if signals.get("notice_period_days",90) <= 30:
        score += 4

    if signals.get("willing_to_relocate"):
        score += 2

    ####################################################
    # 9. Penalize Research-only Profiles
    ####################################################

    title = profile.get("current_title","").lower()

    if "research scientist" in title:
        score -= 8

    ####################################################
    # 10. Penalize CV-only Profiles
    ####################################################

    cv_words = [
        "computer vision",
        "object detection",
        "segmentation"
    ]

    cv_hits = sum(word in text for word in cv_words)

    ir_hits = retrieval_hits

    if cv_hits > 3 and ir_hits == 0:
        score -= 12

    ####################################################
    # Final
    ####################################################

    score = round(score,2)

    results.append(
        (
            score,
            cid,
            profile.get("current_title",""),
            "; ".join(reasons)
        )
    )

####################################################
# Human Recruiter Ranking
####################################################

results = sorted(
    results,
    key=lambda x: (-x[0], x[1])
)

print("Top 20")

for i,r in enumerate(results[:20],1):
    print(i,r)

Top 20
1 (np.float32(193.63), 'CAND_0046064', 'Senior NLP Engineer', 'Very strong semantic match; Ideal experience; Product company experience; Production ML systems; Retrieval and ranking systems; Evaluation framework experience')
2 (np.float32(182.05), 'CAND_0018499', 'Senior Machine Learning Engineer', 'Very strong semantic match; Ideal experience; Product company experience; Production ML systems; Retrieval and ranking systems; Evaluation framework experience')
3 (np.float32(179.78), 'CAND_0081846', 'Lead AI Engineer', 'Very strong semantic match; Ideal experience; Product company experience; Production ML systems; Retrieval and ranking systems; Evaluation framework experience')
4 (np.float32(179.5), 'CAND_0055905', 'Senior Machine Learning Engineer', 'Strong semantic match; Ideal experience; Product company experience; Production ML systems; Retrieval and ranking systems; Evaluation framework experience')
5 (np.float32(177.33), 'CAND_0002025', 'Senior AI Engineer', 'Very strong se

In [ ]:
top200 = results[:200]

print("Top 200 Ready")
print(len(top200))

Top 200 Ready
200


In [ ]:
candidate_lookup = {}

for c in candidates:
    candidate_lookup[c["candidate_id"]] = c

print(len(candidate_lookup))

100000


In [ ]:
reranked = []

for score, cid, title, reason in top200:

    c = candidate_lookup[cid]

    profile = c.get("profile", {})
    history = c.get("career_history", [])
    signals = c.get("redrob_signals", {})
    skills = c.get("skills", [])

    recruiter_score = 0

    ########################################################
    # Combine complete profile text
    ########################################################

    text = ""

    text += profile.get("current_title","") + " "
    text += profile.get("headline","") + " "
    text += profile.get("summary","") + " "

    for s in skills:
        if isinstance(s, dict):
            text += s.get("name","") + " "
        else:
            text += str(s) + " "

    for h in history:
        text += h.get("title","") + " "
        text += h.get("description","") + " "

    text = text.lower()

    ########################################################
    # Core JD Technologies
    ########################################################

    core_keywords = [
        "retrieval","ranking","semantic search",
        "recommendation","embeddings",
        "vector search","hybrid retrieval",
        "bm25","faiss","pinecone",
        "qdrant","milvus",
        "elasticsearch","opensearch",
        "learning to rank",
        "llm","rag"
    ]

    recruiter_score += sum(k in text for k in core_keywords) * 3

    ########################################################
    # Evaluation Framework
    ########################################################

    eval_keywords = [
        "ndcg",
        "mrr",
        "map",
        "recall",
        "precision",
        "evaluation",
        "offline",
        "online",
        "a/b",
        "ab testing"
    ]

    recruiter_score += sum(k in text for k in eval_keywords) * 4

    ########################################################
    # Fine Tuning
    ########################################################

    finetune = [
        "lora",
        "qlora",
        "peft",
        "llama",
        "mistral",
        "fine-tuning",
        "fine tuned"
    ]

    recruiter_score += sum(k in text for k in finetune) * 3

    ########################################################
    # Production Experience
    ########################################################

    production = [
        "production",
        "deployed",
        "serving",
        "latency",
        "real users",
        "million",
        "pipeline",
        "kubernetes",
        "monitoring"
    ]

    recruiter_score += sum(k in text for k in production) * 2

    ########################################################
    # Product Mindset
    ########################################################

    product = [
        "product",
        "customer",
        "recruiter",
        "marketplace",
        "search",
        "recommendation"
    ]

    recruiter_score += sum(k in text for k in product)

    ########################################################
    # Experience
    ########################################################

    exp = profile.get("years_of_experience",0)

    if 5 <= exp <= 9:
        recruiter_score += 15

    elif 4 <= exp < 5:
        recruiter_score += 8

    elif 9 < exp <= 12:
        recruiter_score += 5

    elif exp > 12:
        recruiter_score -= 5

    ########################################################
    # Recruiter Signals
    ########################################################

    if signals.get("open_to_work_flag"):
        recruiter_score += 5

    if signals.get("recruiter_response_rate",0) > 0.7:
        recruiter_score += 4

    if signals.get("github_activity_score",0) > 60:
        recruiter_score += 3

    if signals.get("saved_by_recruiters_30d",0) > 20:
        recruiter_score += 3

    if signals.get("interview_completion_rate",0) > 0.8:
        recruiter_score += 3

    if signals.get("notice_period_days",90) <= 30:
        recruiter_score += 2

    if signals.get("willing_to_relocate"):
        recruiter_score += 2

    ########################################################
    # Final Combined Score
    ########################################################

    final_score = score * 0.45 + recruiter_score * 0.55

    reranked.append(
        (
            round(final_score,2),
            cid,
            title,
            recruiter_score,
            reason
        )
    )

print("Reranking Complete:", len(reranked))

Reranking Complete: 200


In [ ]:
reranked = sorted(
    reranked,
    key=lambda x: (-x[0], x[1])
)

print("TOP 20 AFTER RERANKING\n")

for i, r in enumerate(reranked[:20], 1):
    print(i, r)

TOP 20 AFTER RERANKING

1 (np.float32(161.93), 'CAND_0046064', 'Senior NLP Engineer', 136, 'Very strong semantic match; Ideal experience; Product company experience; Production ML systems; Retrieval and ranking systems; Evaluation framework experience')
2 (np.float32(149.1), 'CAND_0081846', 'Lead AI Engineer', 124, 'Very strong semantic match; Ideal experience; Product company experience; Production ML systems; Retrieval and ranking systems; Evaluation framework experience')
3 (np.float32(149.02), 'CAND_0018499', 'Senior Machine Learning Engineer', 122, 'Very strong semantic match; Ideal experience; Product company experience; Production ML systems; Retrieval and ranking systems; Evaluation framework experience')
4 (np.float32(147.88), 'CAND_0055905', 'Senior Machine Learning Engineer', 122, 'Strong semantic match; Ideal experience; Product company experience; Production ML systems; Retrieval and ranking systems; Evaluation framework experience')
5 (np.float32(143.05), 'CAND_0002025', 

In [ ]:
final_top100 = reranked[:100]

print("Top 100 Ready:", len(final_top100))

Top 100 Ready: 100


In [ ]:
import pandas as pd

rows = []

for rank, (final_score, cid, title, recruiter_score, reason) in enumerate(final_top100, start=1):

    rows.append({
        "candidate_id": cid,
        "rank": rank,
        "score": round(float(final_score), 4),
        "reasoning": reason
    })

submission_df = pd.DataFrame(rows)

submission_df.head()

,candidate_id,rank,score,reasoning
0,CAND_0046064,1,161.93,Very strong semantic match; Ideal experience; ...
1,CAND_0081846,2,149.10,Very strong semantic match; Ideal experience; ...
2,CAND_0018499,3,149.02,Very strong semantic match; Ideal experience; ...
3,CAND_0055905,4,147.88,Strong semantic match; Ideal experience; Produ...
4,CAND_0002025,5,143.05,Very strong semantic match; Ideal experience; ...


In [ ]:
submission_df.to_csv(
    "/content/drive/MyDrive/Colab Notebooks/submission.csv",
    index=False
)

print("submission.csv saved successfully.")

submission.csv saved successfully.


In [ ]:
%%writefile validate_submission.py

import csv
import re
import sys
from pathlib import Path

REQUIRED_HEADER = ["candidate_id", "rank", "score", "reasoning"]
CANDIDATE_ID_PATTERN = re.compile(r"^CAND_[0-9]{7}$")
DATA_ROW_START = 2
EXPECTED_DATA_ROWS = 100


def validate_submission(csv_path):
    errors = []
    path = Path(csv_path)

    if path.suffix.lower() != ".csv":
        errors.append("Filename must use a .csv extension.")
    elif not path.stem:
        errors.append("Filename must be your registered participant ID (e.g. team_xxx.csv).")

    try:
        with open(path, "r", encoding="utf-8", newline="") as f:
            reader = csv.reader(f)

            try:
                header = next(reader)
            except StopIteration:
                errors.append("Row 1 must be the header row; file is empty.")
                return errors

            # Row 1: column names and their order come from this line only
            if header != REQUIRED_HEADER:
                errors.append(
                    "Row 1 (header) must be exactly:\n"
                    f"  {','.join(REQUIRED_HEADER)}\n"
                    f"Found:\n"
                    f"  {','.join(header)}"
                )

            data_rows = []
            for row in reader:
                if any(cell.strip() for cell in row):
                    data_rows.append(row)

    except UnicodeDecodeError:
        errors.append("File must be UTF-8 encoded.")
        return errors
    except OSError as e:
        errors.append(f"Cannot read file: {e}")
        return errors

    n = len(data_rows)
    if n != EXPECTED_DATA_ROWS:
        errors.append(
            f"After the header (row 1), there must be exactly {EXPECTED_DATA_ROWS} "
            f"data rows (rows {DATA_ROW_START}–{DATA_ROW_START + EXPECTED_DATA_ROWS - 1}); "
            f"found {n}."
        )

    seen_ids = set()
    seen_ranks = set()
    by_rank = []

    for i, cells in enumerate(data_rows):
        row_num = DATA_ROW_START + i

        if len(cells) != len(REQUIRED_HEADER):
            errors.append(
                f"Row {row_num}: expected {len(REQUIRED_HEADER)} columns "
                f"({','.join(REQUIRED_HEADER)}), got {len(cells)}."
            )
            continue

        row = dict(zip(REQUIRED_HEADER, cells))
        cid = row["candidate_id"].strip()
        rank_s = row["rank"].strip()
        score_s = row["score"].strip()

        if not cid:
            errors.append(f"Row {row_num}: candidate_id is required.")
        elif not CANDIDATE_ID_PATTERN.match(cid):
            errors.append(
                f"Row {row_num}: candidate_id must be CAND_XXXXXXX (7 digits)."
            )
        elif cid in seen_ids:
            errors.append(f"Row {row_num}: duplicate candidate_id '{cid}'.")
        else:
            seen_ids.add(cid)

        try:
            rank = int(rank_s)
            if str(rank) != rank_s:
                raise ValueError
            if not 1 <= rank <= 100:
                errors.append(f"Row {row_num}: rank must be between 1 and 100.")
            elif rank in seen_ranks:
                errors.append(f"Row {row_num}: duplicate rank {rank}.")
            else:
                seen_ranks.add(rank)
        except ValueError:
            errors.append(f"Row {row_num}: rank must be an integer (1–100).")
            rank = None

        try:
            score = float(score_s)
        except ValueError:
            errors.append(f"Row {row_num}: score must be a float.")
            score = None

        if rank is not None and score is not None and cid:
            by_rank.append((rank, score, cid))

    missing = set(range(1, 101)) - seen_ranks
    if missing:
        errors.append(
            f"Each rank 1–100 must appear exactly once; missing: {sorted(missing)}"
        )

    by_rank.sort(key=lambda x: x[0])

    for i in range(len(by_rank) - 1):
        r1, s1, _ = by_rank[i]
        r2, s2, _ = by_rank[i + 1]
        if s1 < s2:
            errors.append(
                f"score must be non-increasing by rank: "
                f"rank {r1} ({s1}) < rank {r2} ({s2})."
            )

    for i in range(len(by_rank) - 1):
        r1, s1, c1 = by_rank[i]
        r2, s2, c2 = by_rank[i + 1]
        if s1 == s2 and c1 > c2:
            errors.append(
                f"Equal scores at ranks {r1} and {r2}: "
                f"tie-break requires candidate_id ascending "
                f"({c1!r} > {c2!r})."
            )

    return errors


def main():
    if len(sys.argv) != 2:
        print("Usage: python validate_submission.py <participant_id>.csv")
        sys.exit(1)

    errors = validate_submission(sys.argv[1])
    if errors:
        print(f"Validation failed ({len(errors)} issue(s)):\n")
        for e in errors:
            print(f"- {e}")
        sys.exit(1)

    print("Submission is valid.")


if __name__ == "__main__":
    main()


Overwriting validate_submission.py


In [ ]:
!python /content/validate_submission.py "/content/drive/MyDrive/Colab Notebooks/submission.csv"

Submission is valid.
